In [2]:
# Set Project Root
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

%reload_ext autoreload
%autoreload 2

/teamspace/studios/this_studio/Airport-AI


In [2]:
# Verify datasets file
for folder in [
    "datasets/airport/train/images",
    "datasets/airport/train/labels",
    "datasets/airport/valid/images",
    "datasets/airport/valid/labels",
    "datasets/airport/test/images",
    "datasets/airport/test/labels",
]:
    p = PROJECT_ROOT / folder
    print(p)
    print("Exists:", p.exists())
    if p.exists():
        print("Files:", len(list(p.iterdir())))
    print()

/teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images
Exists: True
Files: 5676

/teamspace/studios/this_studio/Airport-AI/datasets/airport/train/labels
Exists: True
Files: 5676

/teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/images
Exists: True
Files: 811

/teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/labels
Exists: True
Files: 811

/teamspace/studios/this_studio/Airport-AI/datasets/airport/test/images
Exists: True
Files: 405

/teamspace/studios/this_studio/Airport-AI/datasets/airport/test/labels
Exists: True
Files: 405



In [3]:
# Verify the dataset
from ultralytics.data.utils import check_det_dataset

check_det_dataset(str(PROJECT_ROOT / "datasets/airport/data.yaml"))

print("Dataset verified successfully.")

Dataset verified successfully.


In [5]:
# Perform a 1-epoch smoke test
from ultralytics import YOLO

model = YOLO(PROJECT_ROOT / "models/yolo26n.pt")

model.train(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    device=0,
    cache=True,
)

New https://pypi.org/project/ultralytics/8.4.112 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/Airport-AI/datasets/airport/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/teamspace/studios/this_studi

 23        [16, 19, 22]  1    243516  ultralytics.nn.modules.head.Detect           [6, 1, True, [64, 128, 256]]  
YOLO26n summary: 260 layers, 2,506,140 parameters, 2,506,140 gradients, 5.8 GFLOPs

Transferred 606/708 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 720.2±213.5 MB/s, size: 55.3 KB)
train: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/labels... 5676 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5676/5676 1.0Kit/s 5.6s0.1ss
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/01_sample_000027_jpg.rf.d664ea3b99ace9605035f470dc26fb14.jpg: 1 duplicate labels removed
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/01_sample_000032_jpg.rf.20a27777446d6500d69936d3a6fe40d0.jpg: 1 duplicate labels removed
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79dfa02026f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [7]:
# Full Model training
model = YOLO(PROJECT_ROOT / "models/yolo26n.pt")

results = model.train(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",

    epochs=150,
    imgsz=640,

    batch=16,
    workers=8,
    device=0,

    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=5e-4,

    warmup_epochs=3,
    patience=25,

    cache=True,
    amp=True,

    pretrained=True,

    project=PROJECT_ROOT / "runs/train",
    name="airport_yolo26n",

    save=True,
    save_period=True,

    val=True,
    plots=True
)

New https://pypi.org/project/ultralytics/8.4.112 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/Airport-AI/datasets/airport/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/teamspace/studios/this_s

train: Caching images (6.5GB RAM): 100% ━━━━━━━━━━━━ 5676/5676 935.2it/s 6.1s<0.0s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 57.4±70.6 MB/s, size: 54.2 KB)
val: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/labels.cache... 811 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 811/811 26.6Mit/s 0.0s
val: /teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/images/03_sample_000856_jpg.rf.a7c93582290dff5623e3db6dfab36308.jpg: 1 duplicate labels removed
val: /teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/images/04_sample_000008_jpg.rf.1ce383715e57d657f93b48f75dcfe68c.jpg: 1 duplicate labels removed
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alte

In [8]:
# Model Evaluation
from ultralytics import YOLO

model = YOLO(PROJECT_ROOT / "runs/train/airport_yolo26n/weights/best.pt")

metrics = model.val(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",
    split="test",
    imgsz=640,
)

print(metrics)

Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 111.1±50.4 MB/s, size: 56.6 KB)
val: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/labels... 405 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 405/405 859.6it/s 0.5s0.1s
val: /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/images/03_sample_000728_jpg.rf.ea536d3f6f5bea80f16a9901c5ac639e.jpg: 1 duplicate labels removed
val: New cache created: /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.0it/s 8.8s0.1s
                   all        405       3801      0.803       0.76      0.774      0.585
              aircraft        405        989      0.901      0.683       0.74      0.634
      

In [3]:
# Export
from ultralytics import YOLO

model = YOLO(PROJECT_ROOT / "runs/train/airport_yolo26n/weights/best.pt")

model.export(format="onnx")

Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CPU (Intel Xeon CPU @ 2.30GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/


YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/teamspace/studios/this_studio/Airport-AI/runs/train/airport_yolo26n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "/home/zeus/miniconda3/envs/cloudspace/bin/python" "onnxslim>=0.1.82"  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ Retry 2/2 failed: Command 'uv pip install --no-cache-dir --python "/home/zeus/miniconda3/envs/cloudspace/bin/python" "onnxslim>=0.1.82"  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ requirements: ❌ Command 'uv pip install --no-cache-dir --python "/home/zeus/miniconda3/envs/cloudspace/bin/python" "onnxslim>=0.1.82"  --i

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


WARNING ⚠️ ONNX: simplifier failure: No module named 'onnxslim'
ONNX: export success ✅ 10.1s, saved as '/teamspace/studios/this_studio/Airport-AI/runs/train/airport_yolo26n/weights/best.onnx' (9.3 MB)

Export complete (19.6s)
Results saved to /teamspace/studios/this_studio/Airport-AI/runs/train/airport_yolo26n/weights/best.onnx
Predict:         yolo predict task=detect model=/teamspace/studios/this_studio/Airport-AI/runs/train/airport_yolo26n/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/teamspace/studios/this_studio/Airport-AI/runs/train/airport_yolo26n/weights/best.onnx imgsz=640 data=/teamspace/studios/this_studio/Airport-AI/datasets/airport/data.yaml  
Visualize:       https://netron.app


'/teamspace/studios/this_studio/Airport-AI/runs/train/airport_yolo26n/weights/best.onnx'

In [ ]:
# Test on Airport Video
# cap = cv2.VideoCapture("data/videos/01_sample.mp4")
# while True:
#     ret, frame = cap.read()
#     if not ret: break
#     result = model(frame)
#     annotated=result[0].plot()
#     cv2.imshow("Airport AI", annotated)
#     if cv2.waitKey(1) == 27: break

In [ ]:
# Export Model
# model.export(format="onnx") # Output: best.onnx